In [1]:
import pandas as pd
import sqlite3

# SQL 실습 준비하기

In [3]:
# 1. 데이터 불러오기 (기존 타이타닉 데이터)
df = pd.read_csv('train.csv')
# 2. 파이썬 안에 가상의 SQL 데이터베이스 연결  
conn = sqlite3.connect(':memory:')
# 3. 판다스 데이터프레임을 SQL 테이블로 저장
df.to_sql('titanic', conn, index=False, if_exists='replace')
# 4. SQL 쿼리를 실행해주는 함수만들기 (편의용)
def run_query(query):
    return pd.read_sql_query(query, conn)

>**코드 해석**<br>
>`conn = sqlite3.connect(':memory:')`<br>
**해석**: "내 컴퓨터 메모리(RAM) 안에 임시 도서관을 하나 지어줘."<br>
**설명**: sqlite3는 파이썬에 내장된 가벼운 데이터베이스 시스템입니다. :memory:라고 적으면 하드디스크에 파일을 만들지 않고, 프로그램을 끄면 사라지는 번개 같은 임시 DB를 만들겠다는 뜻입니다.<br>
`df.to_sql('titanic', conn, index=False, if_exists='replace')`<br>
**해석**: "판다스에 있는 df 데이터를 도서관 안에 'titanic'이라는 이름의 책장으로 옮겨줘."<br>
**설명**:
'titanic': SQL에서 사용할 테이블 이름입니다.<br>
index=False: 판다스의 행 번호(0, 1, 2...)는 데이터가 아니니 옮기지 말라는 뜻입니다.<br>
if_exists='replace': 혹시 이미 똑같은 이름의 책장이 있으면, 새로 덮어쓰라는 뜻입니다.<br>
`def run_query(query): return pd.read_sql_query(query, conn)` <br>
**해석**: "매번 길게 말하기 힘드니까, '질문(query)'만 던지면 바로 답(결과)을 가져오는 통역사를 고용할게."<br>
**설명**: 원래 SQL 결과를 보려면 코드가 긴데, 이걸 run_query라는 함수로 만들어둔 것입니다. 앞으로는 run_query("SQL문")만 치면 결과가 판다스 표 형태로 예쁘게 출력됩니다.<br>

# 판다스를 SQL로 번역하기

## 1. 데이터 살펴보기 (SELECT)
- Pandas: df.head(5)

In [13]:
query = "SELECT * FROM titanic LIMIT 5"
run_query(query)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,None,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,None,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,None,S


## 2. 조건에 맞는 데이터 필터링 (WHERE)
- Pandas: df[(df['Pclass'] == 1) & (df['Sex'] == 'female')]

In [6]:
query = """
SELECT *
FROM titanic
WHERE Pclass = 1 AND Sex = 'female'
"""
run_query(query)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
1,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
2,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S
3,32,1,1,"Spencer, Mrs. William Augustus (Marie Eugenie)",female,NaN,1,0,PC 17569,146.5208,B78,C
4,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C
...,...,...,...,...,...,...,...,...,...,...,...,...
89,857,1,1,"Wick, Mrs. George Dennick (Mary Hitchcock)",female,45.0,1,1,36928,164.8667,None,S
90,863,1,1,"Swift, Mrs. Frederick Joel (Margaret Welles Ba...",female,48.0,0,0,17466,25.9292,D17,S
91,872,1,1,"Beckwith, Mrs. Richard Leonard (Sallie Monypeny)",female,47.0,1,1,11751,52.5542,D35,S
92,880,1,1,"Potter, Mrs. Thomas Jr (Lily Alexenia Wilson)",female,56.0,0,1,11767,83.1583,C50,C


## 3. 그룹별 통계량 계산 (GROUP BY)
- Pandas: df.groupby('Pclass')['Fare'].agg(['mean', 'count'])

In [10]:
query = """
SELECT Pclass,
    AVG(Fare) AS Avg_Fare,
    COUNT(*) AS Passenger_Count
FROM titanic
GROUP BY Pclass
"""
run_query(query)

,Pclass,Avg_Fare,Passenger_Count
0,1,84.154687,216
1,2,20.662183,184
2,3,13.675550,491


## 4. 데이터 정렬 (ORDER BY)
- Pandas : df.sort_values('Fare', ascending=False).head(10)

In [12]:
query="""
SELECT Name, Fare
FROM titanic
ORDER BY Fare DESC
LIMIT 10
"""
run_query(query)

,Name,Fare
0,"Ward, Miss. Anna",512.3292
1,"Cardeza, Mr. Thomas Drake Martinez",512.3292
2,"Lesurer, Mr. Gustave J",512.3292
3,"Fortune, Mr. Charles Alexander",263.0000
4,"Fortune, Miss. Mabel Helen",263.0000
5,"Fortune, Miss. Alice Elizabeth",263.0000
6,"Fortune, Mr. Mark",263.0000
7,"Ryerson, Miss. Emily Borie",262.3750
8,"Ryerson, Miss. Susan Parker ""Suzette""",262.3750
9,"Baxter, Mr. Quigg Edmond",247.5208


## 5. 새로운 범주 만들기 (CASE WHEN)
- Pandas : apply, where

In [15]:
query = """
SELECT Name, Age,
    CASE WHEN Age < 18 THEN 'Child'
        ELSE 'Adult'
    END AS Age_group
FROM titanic
LIMIT 10
"""
run_query(query)

,Name,Age,Age_group
0,"Braund, Mr. Owen Harris",22.0,Adult
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,Adult
2,"Heikkinen, Miss. Laina",26.0,Adult
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,Adult
4,"Allen, Mr. William Henry",35.0,Adult
5,"Moran, Mr. James",NaN,Adult
6,"McCarthy, Mr. Timothy J",54.0,Adult
7,"Palsson, Master. Gosta Leonard",2.0,Child
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",27.0,Adult
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.0,Child


# 6. 여러 테이블 합치기 (JOIN)

In [18]:
# 1. 가상의 등급 설명 테이블 만들기 
pclass_desc = pd.DataFrame({
    'Pclass' : [1, 2, 3],
    'Description' : ['First Class (Upper)', 'Second Class (Middle)', 'Third Class (Lower)']
})
pclass_desc.to_sql('class_info', conn, index=False, if_exists='replace')

# 2. JOIN 쿼리 실행
query = """
SELECT t.Name, t.Pclass, c.Description
FROM titanic t
JOIN class_info c ON t.Pclass = c.Pclass
LIMIT 5
"""
run_query(query)

,Name,Pclass,Description
0,"Braund, Mr. Owen Harris",3,Third Class (Lower)
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,First Class (Upper)
2,"Heikkinen, Miss. Laina",3,Third Class (Lower)
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,First Class (Upper)
4,"Allen, Mr. William Henry",3,Third Class (Lower)


### [SQL 학습 노트]
1. **데이터 추출의 기초**: `SELECT`, `FROM`, `WHERE` 문법을 통해 판다스의 필터링 기능을 SQL로 구현하는 법을 익힘.
2. **데이터 요약 및 집계**: `GROUP BY`와 집계 함수(`AVG`, `COUNT`)를 조합하여 대량의 데이터에서 필요한 통계량을 빠르게 산출함.
3. **가상 컬럼 생성**: `CASE WHEN` 문법을 활용하여 특정 조건(나이 등)에 따른 파생 변수를 SQL 단계에서 바로 생성하는 법을 배움.
4. **테이블 결합 (JOIN)**: 서로 다른 테이블을 공통 키(`Pclass`)를 기준으로 결합하는 `JOIN` 문법을 학습함.